# Categorize Responses in the Behavioural Set

In [ ]:
import sys, os

PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(),".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0,PARENT_DIR)

In [ ]:
sys.path

In [ ]:
import torch
import json
from typing import List, Literal, Optional, Union, Dict
from enum import Enum
from collections import Counter
import datetime
from dataclasses import dataclass
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Local imports
from config import settings
from gcp_utils import download_from_gcs
from inference.v01.inference_utils import predict_word_level, word_labels_to_spans
from error_analysis.error_categorization import ErrorCategorizer
from error_analysis.error_taxonomy import BehaviouralExample

In [ ]:
MODEL_NAME= "dmis-lab/biobert-base-cased-v1.1"
RUN_IDX ="2"# BEST RUN
VERSION = "v01"
# inference pipeline version may change as we improve and modify the pipeline
INFERENCE_PIPELINE_VERSION = "v01" 


## **Load Required Data Files**

In [ ]:
## Load Required Data Files

# Load id2label mapping
print("📂 Loading id2label mapping...")
with open("data/id2label.json", "r") as f:
    id2label = json.load(f)

# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}

print(f"✅ Loaded {len(id2label)} label mappings")
print(f"Labels: {id2label}")

# Load behavioural evaluation set
print("\n📂 Loading behavioural evaluation set...")
with open("behavioural_set.json", "r") as f:
    behavioural_set = json.load(f)

print(f"✅ Loaded {len(behavioural_set)} categories")
print(f"Categories: {list(behavioural_set.keys())}")

# Count total examples
total_examples = sum(len(examples) for examples in behavioural_set.values())
print(f"Total test examples: {total_examples}")

## **Load model from local directory or GCS**

In [ ]:
# Load the model
GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{RUN_IDX}"
BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"

# Create a local directory path for the model
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"

# Check if model already exists locally
if os.path.exists(LOCAL_MODEL_DIR) and os.path.isfile(os.path.join(LOCAL_MODEL_DIR, "config.json")):
    print(f"✅ Model found locally at {LOCAL_MODEL_DIR}")
    print("Skipping download from GCS.")
else:
    # Download the model directory from GCS if not found locally
    print(f"📥 Model not found locally. Downloading from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
    downloaded_path = download_from_gcs(
        gcs_path=GCS_MODEL_PATH,
        local_path=LOCAL_MODEL_DIR,
        bucket_name=BUCKET_NAME
    )
    if downloaded_path:
        print(f"✅ Download complete. Model saved to {LOCAL_MODEL_DIR}")

# Load the model
print(f"📂 Loading model from {LOCAL_MODEL_DIR}...")
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)

# Move to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
print(f"Using device: {device}")
model.to(device)
model.eval()
print("✅ Model is ready to be used")

In [ ]:
LOCAL_MODEL_DIR

## Load the Behavioural Set

In [ ]:
# Fit the behavioural set into the dataclasses
for k,examples in behavioural_set.items():
    for i,ex in enumerate(examples):
        behavioural_set[k][i] = BehaviouralExample(**ex)

In [ ]:
behavioural_set

## Instantiate the Error Categorizer

In [ ]:
error_categorizer = ErrorCategorizer()

## Single Sample Example

In [ ]:
print("Working example:")
my_example = behavioural_set['Long, realistic clinical sentences (THIS IS GOLD 🥇)'][0]
text = my_example.example
print(f"\t{text}")
tokens, token_labels, word_ids, words, word_labels, word_offsets = predict_word_level(
        text=text,
        model=model,
        tokenizer=tokenizer,
        id2label=id2label,
        device=device,
    )
spans = word_labels_to_spans(text=text, word_offsets=word_offsets, word_labels=word_labels)
print("SPANS:")
for s in spans:
    print(f"\t{s}")

*Snipped to check if the expected entities are in the returned spans*

In [ ]:
my_example.entities_with_labels

In [ ]:

# See which entities from the behavioural examples are in:
# For present entities, we also track which span index they were found in
present_with_indices = [(e, error_categorizer._is_entity_in_spans(e["ent"], spans)) for e in my_example.entities_with_labels]
present = [(e,idx) for e, idx in present_with_indices if idx is not None]
missing = [(e,idx) for e, idx in present_with_indices if idx is None]

print(f"present_with_indices:\n\t{present_with_indices}")
print(f"present:\n\t{present}")
print(f"missing:\n\t{missing}")

*Analyze present errors for thi single example*

In [ ]:
errors = error_categorizer._check_entity_detection(
    example=my_example,
    spans=spans
)

present_errors = errors.get("present_errors", [])
missing_entities = errors.get("missing_entities", [])
false_positives = errors.get("false_positives", [])

print("============== ERRORS FOUND ==============")

if present_errors:
    print("\n--- Present errors (detected but wrong boundary/label) ---")
    for p in present_errors:
        e = p['entity']   # ground truth
        s = p['span']     # prediction
        print(f"  Ground Truth : {e['ent']} ({e['label']}) [{e['start']}:{e['end']}]")
        print(f"  Prediction   : {s['text']} ({s['label']}) [{s['start']}:{s['end']}]")
        print(f"  Error(s)     : {p['errors']}\n")

if missing_entities:
    print("\n--- Missing entities (not detected at all) ---")
    for p in missing_entities:
        e = p['entity']   # ground truth only — no predicted span
        print(f"  Ground Truth : {e['ent']} ({e['label']}) [{e['start']}:{e['end']}]")
        print(f"  Prediction   : (not detected)")
        print(f"  Error(s)     : {p['errors']}\n")

if false_positives:
    print("\n--- False positives (predicted but not expected) ---")
    for p in false_positives:
        s = p['span']     # prediction only — no matching ground truth
        print(f"  Prediction   : {s['text']} ({s['label']}) [{s['start']}:{s['end']}]")
        print(f"  Ground Truth : (none — unexpected prediction)")
        print(f"  Error(s)     : {p['errors']}\n")

if not any([present_errors, missing_entities, false_positives]):
    print("  No errors found.")

# Run Error Categorization thoughout the entire dataset

In [ ]:
def _run_through_behavioural_set(
    error_categorizer: ErrorCategorizer,
    behavioural_set: Dict[str, List[BehaviouralExample]],
    model,
    tokenizer,
    id2label: dict,
    device: str,
) -> dict:
    """Run inference + `_check_entity_detection` on every example.

    Clears `error_categorizer.error_counts` so one full sweep has a single aggregate counter.
    Returns dict with `error_counts` (Counter), plus flat lists of error records for inspection.
    """
    error_categorizer.error_counts.clear()

    total_present_errors: List[dict] = []
    total_missing_entities: List[dict] = []
    total_false_positives: List[dict] = []

    for ex_type, examples in behavioural_set.items():
        print(f"Category: {ex_type}")
        for ex in examples:
            text = ex.example
            tokens, token_labels, word_ids, words, word_labels, word_offsets = predict_word_level(
                text=text,
                model=model,
                tokenizer=tokenizer,
                id2label=id2label,
                device=device,
            )
            spans = word_labels_to_spans(
                text=text,
                word_offsets=word_offsets,
                word_labels=word_labels,
            )
            errors = error_categorizer._check_entity_detection(example=ex, spans=spans)
            pe = errors.get("present_errors", [])
            me = errors.get("missing_entities", [])
            fp = errors.get("false_positives", [])
            total_present_errors.extend(pe)
            total_missing_entities.extend(me)
            total_false_positives.extend(fp)
        print(f"  Processed {len(examples)} examples\n")

    return {
        "error_counts": error_categorizer.error_counts,
        "present_errors": total_present_errors,
        "missing_entities": total_missing_entities,
        "false_positives": total_false_positives,
    }


In [ ]:
results = _run_through_behavioural_set(
    behavioural_set=behavioural_set,
    error_categorizer=error_categorizer,
    model=model,
    id2label=id2label,
    tokenizer=tokenizer,
    device=device
)

In [ ]:
# These results are very important they dictate the next steps for improving the model!
results["error_counts"]


In [ ]:
results

In [ ]:
{'error_counts': Counter({'Irrelevant Span Mislabeling': 75,
          'Model Overgeneralization': 74,
          'Missing Expected Entity': 11,
          'Boundary Overreach': 8,
          'Tokenization Artifacts': 4,
          'BIO Sequencing Errors': 4,
          'Incorrect Polarity Assignment': 3}),
 'present_errors': [{'entity': {'ent': 'bradypnea',
    'label': 'SYMPTOM_POS',
    'start': 13,
    'end': 22},
   'span': {'start': 13,
    'end': 22,
    'text': 'bradypnea',
    'label': 'CONFLICT-B-SYMPTOM_NEG-I-SYMPTOM_POS-I-SYMPTOM_NEG-I-SYMPTOM_NEG'},
   'span_idx': 2,
   'errors': ['Tokenization Artifacts', 'BIO Sequencing Errors']},
  {'entity': {'ent': 'dysphonia',
    'label': 'SYMPTOM_POS',
    'start': 13,
    'end': 22},
   'span': {'start': 13,
    'end': 22,
    'text': 'dysphonia',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 1,
   'errors': ['Incorrect Polarity Assignment']},
  {'entity': {'ent': 'wheezing',
    'label': 'SYMPTOM_POS',
    'start': 30,
    'end': 38},
   'span': {'start': 30,
    'end': 38,
    'text': 'wheezing',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 5,
   'errors': ['Incorrect Polarity Assignment']},
  {'entity': {'ent': 'tetanic convulsion',
    'label': 'SYMPTOM_POS',
    'start': 14,
    'end': 32},
   'span': {'start': 14,
    'end': 33,
    'text': 'tetanic convulsion,',
    'label': 'SYMPTOM_POS'},
   'span_idx': 2,
   'errors': ['Boundary Overreach']},
  {'entity': {'ent': 'ventricular fibrillation',
    'label': 'SYMPTOM_POS',
    'start': 34,
    'end': 58},
   'span': {'start': 34,
    'end': 59,
    'text': 'ventricular fibrillation,',
    'label': 'SYMPTOM_POS'},
   'span_idx': 3,
   'errors': ['Boundary Overreach']},
  {'entity': {'ent': 'multiple sites abdominal mass',
    'label': 'SYMPTOM_POS',
    'start': 64,
    'end': 93},
   'span': {'start': 64,
    'end': 99,
    'text': 'multiple sites abdominal mass since',
    'label': 'SYMPTOM_POS'},
   'span_idx': 5,
   'errors': ['Boundary Overreach']},
  {'entity': {'ent': 'phlegm', 'label': 'SYMPTOM_POS', 'start': 9, 'end': 15},
   'span': {'start': 9,
    'end': 21,
    'text': 'phlegm along',
    'label': 'SYMPTOM_POS'},
   'span_idx': 1,
   'errors': ['Boundary Overreach']},
  {'entity': {'ent': 'fatigue',
    'label': 'SYMPTOM_POS',
    'start': 124,
    'end': 131},
   'span': {'start': 119,
    'end': 131,
    'text': 'mild fatigue',
    'label': 'SYMPTOM_POS'},
   'span_idx': 11,
   'errors': ['Boundary Overreach']},
  {'entity': {'ent': 'fever',
    'label': 'SYMPTOM_NEG',
    'start': 139,
    'end': 144},
   'span': {'start': 139,
    'end': 154,
    'text': 'fever or chills',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 14,
   'errors': ['Boundary Overreach']},
  {'entity': {'ent': 'chills',
    'label': 'SYMPTOM_NEG',
    'start': 148,
    'end': 154},
   'span': {'start': 139,
    'end': 154,
    'text': 'fever or chills',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 14,
   'errors': ['Boundary Overreach']},
  {'entity': {'ent': 'respiratory paralysis',
    'label': 'SYMPTOM_POS',
    'start': 61,
    'end': 82},
   'span': {'start': 48,
    'end': 89,
    'text': 'intermittent respiratory paralysis, which',
    'label': 'SYMPTOM_POS'},
   'span_idx': 8,
   'errors': ['Boundary Overreach']},
  {'entity': {'ent': 'steatorrhea',
    'label': 'SYMPTOM_POS',
    'start': 50,
    'end': 61},
   'span': {'start': 50,
    'end': 61,
    'text': 'steatorrhea',
    'label': 'CONFLICT-B-SYMPTOM_NEG-I-SYMPTOM_POS-I-SYMPTOM_NEG-I-SYMPTOM_NEG-I-SYMPTOM_NEG'},
   'span_idx': 8,
   'errors': ['Tokenization Artifacts', 'BIO Sequencing Errors']},
  {'entity': {'ent': 'anuria', 'label': 'SYMPTOM_POS', 'start': 21, 'end': 27},
   'span': {'start': 21,
    'end': 27,
    'text': 'anuria',
    'label': 'CONFLICT-B-SYMPTOM_POS-I-SYMPTOM_NEG'},
   'span_idx': 3,
   'errors': ['Tokenization Artifacts', 'BIO Sequencing Errors']},
  {'entity': {'ent': 'peripheral muscle weakness',
    'label': 'SYMPTOM_POS',
    'start': 15,
    'end': 41},
   'span': {'start': 15,
    'end': 41,
    'text': 'peripheral muscle weakness',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 3,
   'errors': ['Incorrect Polarity Assignment']},
  {'entity': {'ent': 'vaginismus',
    'label': 'SYMPTOM_NEG',
    'start': 50,
    'end': 60},
   'span': {'start': 50,
    'end': 60,
    'text': 'vaginismus',
    'label': 'CONFLICT-B-SYMPTOM_NEG-I-SYMPTOM_NEG-I-SYMPTOM_POS-I-SYMPTOM_NEG'},
   'span_idx': 8,
   'errors': ['Tokenization Artifacts', 'BIO Sequencing Errors']},
  {'entity': {'ent': 'fixed dilated pupils',
    'label': 'O',
    'start': 33,
    'end': 53},
   'span': {'start': 33,
    'end': 53,
    'text': 'fixed dilated pupils',
    'label': 'SYMPTOM_POS'},
   'span_idx': 3,
   'errors': ['Irrelevant Span Mislabeling']}],
 'missing_entities': [{'entity': {'ent': 'localized superficial lump',
    'label': 'SYMPTOM_POS',
    'start': 16,
    'end': 42},
   'span': None,
   'errors': ['Missing Expected Entity'],
   'reasoning': "Expected entity 'localized superficial lump' was not detected"},
  {'entity': {'ent': 'corkscrew hair',
    'label': 'SYMPTOM_POS',
    'start': 13,
    'end': 27},
   'span': None,
   'errors': ['Missing Expected Entity'],
   'reasoning': "Expected entity 'corkscrew hair' was not detected"},
  {'entity': {'ent': 'bloody diarrhea',
    'label': 'SYMPTOM_NEG',
    'start': 7,
    'end': 22},
   'span': None,
   'errors': ['Missing Expected Entity'],
   'reasoning': "Expected entity 'bloody diarrhea' was not detected"},
  {'entity': {'ent': 'precordial pain',
    'label': 'SYMPTOM_POS',
    'start': 22,
    'end': 37},
   'span': None,
   'errors': ['Missing Expected Entity'],
   'reasoning': "Expected entity 'precordial pain' was not detected"},
  {'entity': {'ent': 'multiple sites pelvic lump',
    'label': 'SYMPTOM_POS',
    'start': 17,
    'end': 43},
   'span': None,
   'errors': ['Missing Expected Entity'],
   'reasoning': "Expected entity 'multiple sites pelvic lump' was not detected"},
  {'entity': {'ent': 'digestive system symptom',
    'label': 'SYMPTOM_POS',
    'start': 25,
    'end': 49},
   'span': None,
   'errors': ['Missing Expected Entity'],
   'reasoning': "Expected entity 'digestive system symptom' was not detected"},
  {'entity': {'ent': 'inability to feed',
    'label': 'SYMPTOM_POS',
    'start': 28,
    'end': 45},
   'span': None,
   'errors': ['Missing Expected Entity'],
   'reasoning': "Expected entity 'inability to feed' was not detected"},
  {'entity': {'ent': 'generalized pelvic mass',
    'label': 'SYMPTOM_POS',
    'start': 29,
    'end': 52},
   'span': None,
   'errors': ['Missing Expected Entity'],
   'reasoning': "Expected entity 'generalized pelvic mass' was not detected"},
  {'entity': {'ent': 'hemorrhage from throat',
    'label': 'SYMPTOM_POS',
    'start': 8,
    'end': 30},
   'span': None,
   'errors': ['Missing Expected Entity'],
   'reasoning': "Expected entity 'hemorrhage from throat' was not detected"},
  {'entity': {'ent': 'progressive prostration',
    'label': 'O',
    'start': 49,
    'end': 72},
   'span': None,
   'errors': ['Missing Expected Entity'],
   'reasoning': "Expected entity 'progressive prostration' was not detected"},
  {'entity': {'ent': 'hepatic dysfunction',
    'label': 'O',
    'start': 45,
    'end': 64},
   'span': None,
   'errors': ['Missing Expected Entity'],
   'reasoning': "Expected entity 'hepatic dysfunction' was not detected"}],
 'false_positives': [{'entity': None,
   'span': {'start': 16,
    'end': 25,
    'text': 'localized',
    'label': 'SYMPTOM_POS'},
   'span_idx': 2,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'localized' not in expected entities"},
  {'entity': None,
   'span': {'start': 26,
    'end': 68,
    'text': 'superficial lump that started two days ago',
    'label': 'SYMPTOM_POS'},
   'span_idx': 3,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'superficial lump that started two days ago' not in expected entities"},
  {'entity': None,
   'span': {'start': 13,
    'end': 22,
    'text': 'corkscrew',
    'label': 'CONFLICT-B-SYMPTOM_POS-I-SYMPTOM_POS-I-SYMPTOM_NEG-I-SYMPTOM_NEG'},
   'span_idx': 2,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'corkscrew' not in expected entities"},
  {'entity': None,
   'span': {'start': 23, 'end': 27, 'text': 'hair', 'label': 'SYMPTOM_POS'},
   'span_idx': 3,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'hair' not in expected entities"},
  {'entity': None,
   'span': {'start': 29, 'end': 34, 'text': 'which', 'label': 'SYMPTOM_POS'},
   'span_idx': 5,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'which' not in expected entities"},
  {'entity': None,
   'span': {'start': 39,
    'end': 58,
    'text': 'been worsening over',
    'label': 'SYMPTOM_POS'},
   'span_idx': 7,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'been worsening over' not in expected entities"},
  {'entity': None,
   'span': {'start': 63,
    'end': 72,
    'text': 'past week',
    'label': 'SYMPTOM_POS'},
   'span_idx': 9,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'past week' not in expected entities"},
  {'entity': None,
   'span': {'start': 51,
    'end': 65,
    'text': 'began suddenly',
    'label': 'SYMPTOM_POS'},
   'span_idx': 3,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'began suddenly' not in expected entities"},
  {'entity': None,
   'span': {'start': 71, 'end': 78, 'text': 'morning', 'label': 'SYMPTOM_POS'},
   'span_idx': 5,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'morning' not in expected entities"},
  {'entity': None,
   'span': {'start': 31, 'end': 33, 'text': 'on', 'label': 'SYMPTOM_NEG'},
   'span_idx': 3,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'on' not in expected entities"},
  {'entity': None,
   'span': {'start': 34, 'end': 41, 'text': 'and off', 'label': 'SYMPTOM_POS'},
   'span_idx': 4,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'and off' not in expected entities"},
  {'entity': None,
   'span': {'start': 46,
    'end': 60,
    'text': 'several months',
    'label': 'SYMPTOM_POS'},
   'span_idx': 6,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'several months' not in expected entities"},
  {'entity': None,
   'span': {'start': 27,
    'end': 46,
    'text': 'gradual improvement',
    'label': 'SYMPTOM_POS'},
   'span_idx': 3,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'gradual improvement' not in expected entities"},
  {'entity': None,
   'span': {'start': 7, 'end': 13, 'text': 'bloody', 'label': 'SYMPTOM_NEG'},
   'span_idx': 1,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'bloody' not in expected entities"},
  {'entity': None,
   'span': {'start': 14,
    'end': 22,
    'text': 'diarrhea',
    'label': 'CONFLICT-I-SYMPTOM_NEG-I-SYMPTOM_NEG-I-SYMPTOM_POS-I-SYMPTOM_POS'},
   'span_idx': 2,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'diarrhea' not in expected entities"},
  {'entity': None,
   'span': {'start': 42, 'end': 49, 'text': 'related', 'label': 'SYMPTOM_NEG'},
   'span_idx': 7,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'related' not in expected entities"},
  {'entity': None,
   'span': {'start': 54, 'end': 56, 'text': 'it', 'label': 'SYMPTOM_NEG'},
   'span_idx': 8,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'it' not in expected entities"},
  {'entity': None,
   'span': {'start': 22,
    'end': 32,
    'text': 'precordial',
    'label': 'CONFLICT-B-SYMPTOM_NEG-I-SYMPTOM_POS-I-SYMPTOM_NEG'},
   'span_idx': 3,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'precordial' not in expected entities"},
  {'entity': None,
   'span': {'start': 33, 'end': 37, 'text': 'pain', 'label': 'SYMPTOM_NEG'},
   'span_idx': 4,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'pain' not in expected entities"},
  {'entity': None,
   'span': {'start': 17,
    'end': 31,
    'text': 'multiple sites',
    'label': 'SYMPTOM_POS'},
   'span_idx': 3,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'multiple sites' not in expected entities"},
  {'entity': None,
   'span': {'start': 32,
    'end': 38,
    'text': 'pelvic',
    'label': 'CONFLICT-I-SYMPTOM_POS-I-SYMPTOM_NEG-I-SYMPTOM_POS'},
   'span_idx': 4,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'pelvic' not in expected entities"},
  {'entity': None,
   'span': {'start': 39, 'end': 43, 'text': 'lump', 'label': 'SYMPTOM_NEG'},
   'span_idx': 5,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'lump' not in expected entities"},
  {'entity': None,
   'span': {'start': 43, 'end': 44, 'text': ',', 'label': 'SYMPTOM_POS'},
   'span_idx': 6,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span ',' not in expected entities"},
  {'entity': None,
   'span': {'start': 25,
    'end': 34,
    'text': 'digestive',
    'label': 'SYMPTOM_POS'},
   'span_idx': 4,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'digestive' not in expected entities"},
  {'entity': None,
   'span': {'start': 35, 'end': 41, 'text': 'system', 'label': 'SYMPTOM_NEG'},
   'span_idx': 5,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'system' not in expected entities"},
  {'entity': None,
   'span': {'start': 42,
    'end': 49,
    'text': 'symptom',
    'label': 'CONFLICT-I-SYMPTOM_POS-I-SYMPTOM_POS-I-SYMPTOM_NEG-I-SYMPTOM_NEG'},
   'span_idx': 6,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'symptom' not in expected entities"},
  {'entity': None,
   'span': {'start': 100,
    'end': 110,
    'text': 'last night',
    'label': 'SYMPTOM_POS'},
   'span_idx': 6,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'last night' not in expected entities"},
  {'entity': None,
   'span': {'start': 28,
    'end': 37,
    'text': 'inability',
    'label': 'SYMPTOM_POS'},
   'span_idx': 3,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'inability' not in expected entities"},
  {'entity': None,
   'span': {'start': 38,
    'end': 60,
    'text': 'to feed after exertion',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 4,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'to feed after exertion' not in expected entities"},
  {'entity': None,
   'span': {'start': 41,
    'end': 83,
    'text': 'started approximately three days ago after',
    'label': 'SYMPTOM_POS'},
   'span_idx': 5,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'started approximately three days ago after' not in expected entities"},
  {'entity': None,
   'span': {'start': 84,
    'end': 92,
    'text': 'physical',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 6,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'physical' not in expected entities"},
  {'entity': None,
   'span': {'start': 93,
    'end': 101,
    'text': 'exertion',
    'label': 'CONFLICT-I-SYMPTOM_NEG-I-SYMPTOM_POS-I-SYMPTOM_NEG'},
   'span_idx': 7,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'exertion' not in expected entities"},
  {'entity': None,
   'span': {'start': 132, 'end': 135, 'text': 'but', 'label': 'SYMPTOM_NEG'},
   'span_idx': 12,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'but' not in expected entities"},
  {'entity': None,
   'span': {'start': 9,
    'end': 18,
    'text': 'past week',
    'label': 'SYMPTOM_POS'},
   'span_idx': 2,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'past week' not in expected entities"},
  {'entity': None,
   'span': {'start': 96,
    'end': 108,
    'text': 'to worsen in',
    'label': 'SYMPTOM_POS'},
   'span_idx': 10,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'to worsen in' not in expected entities"},
  {'entity': None,
   'span': {'start': 113,
    'end': 125,
    'text': 'evenings and',
    'label': 'SYMPTOM_POS'},
   'span_idx': 12,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'evenings and' not in expected entities"},
  {'entity': None,
   'span': {'start': 126,
    'end': 144,
    'text': 'partially improves',
    'label': 'SYMPTOM_POS'},
   'span_idx': 13,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'partially improves' not in expected entities"},
  {'entity': None,
   'span': {'start': 145, 'end': 149, 'text': 'with', 'label': 'SYMPTOM_NEG'},
   'span_idx': 14,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'with' not in expected entities"},
  {'entity': None,
   'span': {'start': 150, 'end': 154, 'text': 'rest', 'label': 'SYMPTOM_POS'},
   'span_idx': 15,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'rest' not in expected entities"},
  {'entity': None,
   'span': {'start': 18,
    'end': 28,
    'text': 'persistent',
    'label': 'SYMPTOM_POS'},
   'span_idx': 2,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'persistent' not in expected entities"},
  {'entity': None,
   'span': {'start': 29,
    'end': 40,
    'text': 'generalized',
    'label': 'SYMPTOM_POS'},
   'span_idx': 3,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'generalized' not in expected entities"},
  {'entity': None,
   'span': {'start': 41,
    'end': 47,
    'text': 'pelvic',
    'label': 'CONFLICT-I-SYMPTOM_POS-I-SYMPTOM_NEG-I-SYMPTOM_POS'},
   'span_idx': 4,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'pelvic' not in expected entities"},
  {'entity': None,
   'span': {'start': 48, 'end': 52, 'text': 'mass', 'label': 'SYMPTOM_NEG'},
   'span_idx': 5,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'mass' not in expected entities"},
  {'entity': None,
   'span': {'start': 61,
    'end': 75,
    'text': 'clear triggers',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 7,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'clear triggers' not in expected entities"},
  {'entity': None,
   'span': {'start': 107,
    'end': 124,
    'text': 'recent infections',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 15,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'recent infections' not in expected entities"},
  {'entity': None,
   'span': {'start': 10,
    'end': 20,
    'text': 'last visit',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 2,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'last visit' not in expected entities"},
  {'entity': None,
   'span': {'start': 40,
    'end': 49,
    'text': 'worsening',
    'label': 'SYMPTOM_POS'},
   'span_idx': 7,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'worsening' not in expected entities"},
  {'entity': None,
   'span': {'start': 63,
    'end': 86,
    'text': 'especially when walking',
    'label': 'SYMPTOM_POS'},
   'span_idx': 10,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'especially when walking' not in expected entities"},
  {'entity': None,
   'span': {'start': 87, 'end': 91, 'text': 'long', 'label': 'SYMPTOM_NEG'},
   'span_idx': 11,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'long' not in expected entities"},
  {'entity': None,
   'span': {'start': 92,
    'end': 101,
    'text': 'distances',
    'label': 'SYMPTOM_POS'},
   'span_idx': 12,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'distances' not in expected entities"},
  {'entity': None,
   'span': {'start': 73, 'end': 77, 'text': 'been', 'label': 'SYMPTOM_POS'},
   'span_idx': 12,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'been' not in expected entities"},
  {'entity': None,
   'span': {'start': 78,
    'end': 87,
    'text': 'affecting',
    'label': 'SYMPTOM_POS'},
   'span_idx': 13,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'affecting' not in expected entities"},
  {'entity': None,
   'span': {'start': 88, 'end': 93, 'text': 'daily', 'label': 'SYMPTOM_NEG'},
   'span_idx': 14,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'daily' not in expected entities"},
  {'entity': None,
   'span': {'start': 94,
    'end': 117,
    'text': 'activities despite over',
    'label': 'SYMPTOM_POS'},
   'span_idx': 15,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'activities despite over' not in expected entities"},
  {'entity': None,
   'span': {'start': 117, 'end': 118, 'text': '-', 'label': 'SYMPTOM_NEG'},
   'span_idx': 16,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span '-' not in expected entities"},
  {'entity': None,
   'span': {'start': 118, 'end': 121, 'text': 'the', 'label': 'SYMPTOM_POS'},
   'span_idx': 17,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'the' not in expected entities"},
  {'entity': None,
   'span': {'start': 121,
    'end': 129,
    'text': '-counter',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 18,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span '-counter' not in expected entities"},
  {'entity': None,
   'span': {'start': 130,
    'end': 141,
    'text': 'medications',
    'label': 'SYMPTOM_POS'},
   'span_idx': 19,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'medications' not in expected entities"},
  {'entity': None,
   'span': {'start': 46, 'end': 52, 'text': 'recent', 'label': 'SYMPTOM_NEG'},
   'span_idx': 8,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'recent' not in expected entities"},
  {'entity': None,
   'span': {'start': 53,
    'end': 71,
    'text': 'medication changes',
    'label': 'SYMPTOM_POS'},
   'span_idx': 9,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'medication changes' not in expected entities"},
  {'entity': None,
   'span': {'start': 32,
    'end': 56,
    'text': 'increased stress at work',
    'label': 'SYMPTOM_POS'},
   'span_idx': 3,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'increased stress at work' not in expected entities"},
  {'entity': None,
   'span': {'start': 8,
    'end': 23,
    'text': 'hemorrhage from',
    'label': 'SYMPTOM_POS'},
   'span_idx': 1,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'hemorrhage from' not in expected entities"},
  {'entity': None,
   'span': {'start': 24, 'end': 30, 'text': 'throat', 'label': 'SYMPTOM_POS'},
   'span_idx': 2,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'throat' not in expected entities"},
  {'entity': None,
   'span': {'start': 41,
    'end': 55,
    'text': 'recent illness',
    'label': 'SYMPTOM_POS'},
   'span_idx': 4,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'recent illness' not in expected entities"},
  {'entity': None,
   'span': {'start': 48,
    'end': 56,
    'text': 'starting',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 5,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'starting' not in expected entities"},
  {'entity': None,
   'span': {'start': 59,
    'end': 79,
    'text': 'new exercise routine',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 7,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'new exercise routine' not in expected entities"},
  {'entity': None,
   'span': {'start': 37,
    'end': 47,
    'text': 'poor sleep',
    'label': 'SYMPTOM_POS'},
   'span_idx': 5,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'poor sleep' not in expected entities"},
  {'entity': None,
   'span': {'start': 27, 'end': 34, 'text': 'warning', 'label': 'SYMPTOM_NEG'},
   'span_idx': 4,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'warning' not in expected entities"},
  {'entity': None,
   'span': {'start': 35,
    'end': 45,
    'text': 'signs such',
    'label': 'SYMPTOM_POS'},
   'span_idx': 5,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'signs such' not in expected entities"},
  {'entity': None,
   'span': {'start': 49,
    'end': 60,
    'text': 'progressive',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 7,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'progressive' not in expected entities"},
  {'entity': None,
   'span': {'start': 61,
    'end': 72,
    'text': 'prostration',
    'label': 'CONFLICT-I-SYMPTOM_NEG-I-SYMPTOM_POS-I-SYMPTOM_NEG'},
   'span_idx': 8,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'prostration' not in expected entities"},
  {'entity': None,
   'span': {'start': 45,
    'end': 52,
    'text': 'hepatic',
    'label': 'CONFLICT-B-SYMPTOM_POS-I-SYMPTOM_POS-I-SYMPTOM_NEG'},
   'span_idx': 5,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'hepatic' not in expected entities"},
  {'entity': None,
   'span': {'start': 53,
    'end': 64,
    'text': 'dysfunction',
    'label': 'SYMPTOM_POS'},
   'span_idx': 6,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'dysfunction' not in expected entities"},
  {'entity': None,
   'span': {'start': 10,
    'end': 23,
    'text': 'questionnaire',
    'label': 'SYMPTOM_POS'},
   'span_idx': 1,
   'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
   'reasoning': "Predicted entity span 'questionnaire' not in expected entities"}]}

## **Build AI Evaluation Loop** - LATER 

In [ ]:
# from xai_sdk import Client
# from xai_sdk.chat import system, user

# from pydantic import BaseModel

# class DummyInfo(BaseModel):
#     message: str

# client = Client(api_key=os.getenv("XAI_API_KEY"))
# chat = client.chat.create(model="grok-4-1-fast-non-reasoning-latest")


# # test_prompt = "Say hello in a JSON object under the key 'message'."
# # chat.append(user(test_prompt))
# # response, invoice = chat.parse(DummyInfo)

In [ ]:
# # Extract error type keys from ErrorTaxonomy dataclass
# ErrorTypeKeys =ErrorTaxonomy.__dataclass_fields__.keys()
# # Create a set of valid error type keys for validation
# _valid_error_types = set(ErrorTypeKeys)

# # Create an Enum from the valid error types
# ErrorType = Enum('ErrorType', {k: k for k in _valid_error_types})

# class LLMResponse(BaseModel):
#     """
#     Response from LLM categorizing errors in NER predictions.
#     """
#     error_type: Optional[ErrorType] = Field(
#         default=None,
#         description=f"The error category code. Must be one of: {_valid_error_types}"
#     )
#     reasoning: str = Field(
#         description="Explanation of the error categorization, including which spans/entities are problematic and why"
#     )
    

In [ ]:
# SYSTEM_PROMPT = """You are an expert at analyzing Named Entity Recognition (NER) model predictions for clinical symptom extraction.
# Your task is to categorize errors in model predictions by comparing predicted spans against expected entities.
# SPAN Entities: SYMPTOM_POS (a symptom reported by the patient.), SYMPTOM_NEG (a symptom negated by the patient.), O, not an entity

# ## Error Taxonomy
# You must classify errors into ONE of these 8 categories (or return None if no error):

# IMPORTANT: Return the SHORT CODE (type_0, type_1, etc.) NOT the full name!

# - type_0: Irrelevant Span Mislabeling
#   - A non-relevant span is incorrectly detected as an entity
#   - Example: `65` labeled as SYMPTOM_POS
#   - Indicates semantic confusion—model cannot distinguish between entity and non-entity tokens

# - type_1: Missing Expected Entity
#   - A clinically relevant symptom is not detected at all
#   - Example: `exanthema` not extracted
#   - Strong signal for dataset enrichment

# - type_2: Tokenization Artifacts
#   - Errors caused by subword splits influencing predictions
#   - Example: Mixed labels across subwords of a single word, conflicting labels aggregated into `CONFLICT-*`
#   - Expected behavior with WordPiece/BPE tokenizers

# - type_3: Incorrect Polarity Assignment
#   - Incorrect polarity assignment in the presence of negation
#   - Example: `muscle cramps` labeled as `SYMPTOM_NEG` when context implies presence
#   - Model struggles with negation scope and contrastive clauses

# - type_4: BIO Sequencing Errors
#   - Incorrect or inconsistent BIO tag transitions
#   - Example: `I-SYMPTOM_POS` without a preceding `B-`, new symptom starting with `I-` instead of `B-`
#   - Model uncertainty at entity boundaries

# - type_5: Boundary Overreach
#   - Model captures a symptom PLUS unrelated surrounding words
#   - Example: `blisters on his head` (should be just `blisters`)
#   - Core entity detected correctly but span precision is low

# - type_6: Boundary Undereach
#   - Model captures only part of a multi-word symptom
#   - Example: `chest` `pain` instead of `chest pain`
#   - Partial entity detection

# - type_7: Model Overgeneralization
#   - Model predicts a symptom where none exists
#   - Example: Predicting `SYMPTOM_POS` for person names (`Marie`), general states (`well`), demographics (`65`)
#   - Model has learned overly broad symptom cues

# ## Input Format

# You will receive:
# - `input_text`: Original text
# - `tokens`: Tokenized tokens (filtered, no special tokens)
# - `token_level_labels`: BIO labels for each token (e.g., "B-SYMPTOM_POS", "I-SYMPTOM_POS", "O")
# - `word_ids`: Word index for each token (tokens from same word share same word_id)
# - `words`: Naive word split of text
# - `word_level_labels`: Aggregated BIO labels per word
# - `predicted_spans`: List of predicted entity spans with {start, end, text, label}
# - `expected_entities`: List of expected entity text strings (from ground truth)


# ## Your Task

# 1. Compare predicted_spans against expected_entities
# 2. Identify discrepancies (missing entities, extra entities, wrong boundaries, wrong polarity)
# 3. Classify the PRIMARY error type (choose the most significant issue)
# 4. Return the SHORT CODE (type_0, type_1, type_2, etc.) in the error_type field
# 5. Provide clear reasoning explaining:
#    - Which spans/entities are problematic
#    - Why this error type was chosen
#    - What the model got right vs wrong
#    - No more than 30 words per explanation. Be concise.


# - If prediction matches expected perfectly, return error_type=None
# - If multiple error types apply, note all errors
# - CRITICAL: Use the exact short code format (type_0, type_1, etc.) - do NOT use full names
# - Be specific in reasoning—reference actual spans and text
# - Consider tokenization artifacts when spans don't align perfectly
# - Pay attention to BIO tag sequences and polarity (POS vs NEG)"""